---
## Stage 12-13 v2: Probability Calibration & Final Evaluation

**วัตถุประสงค์:**
- Stage 12: ปรับ raw probabilities ให้สะท้อนความน่าจะเป็นจริง (Isotonic Regression)
- Stage 13: เลือก threshold + evaluate บน test set

**Input:** `model.pt`, `scaler.pkl`, `feature_cols.pkl`, `feature_matrix.csv`  
**Output:** `calibrator.pkl`, `test_metrics.json`

### ปรับปรุงจาก v1
- โหลด feature_matrix จาก Stage 9/10 v2
- entity split ใช้ numpy RNG (consistent กับ Stage 10/11 v2)
- เพิ่ม config cell ด้านบน

**กฎสำคัญ:**
- Calibrate บน **validation set** เท่านั้น
- Threshold selection บน **validation set**
- Final evaluation บน **test set** ครั้งเดียว!

| Sub-step | หน้าที่ |
|----------|--------|
| 12.1 | Collect Raw Probabilities (val set) |
| 12.2 | Fit Calibrator (Isotonic Regression) |
| 12.3 | Calibration Quality Check |
| 13.1 | Threshold Selection (val set) |
| 13.2 | 3-Level Threshold Definition |
| 13.3 | Final Evaluation (test set — ครั้งเดียว!) |

In [1]:
# ─── Config ───────────────────────────────────────────────────────────────
OUTPUT_DIR      = '/Users/tm/Documents/GitHub/Project-for-Work/data/processed'
FEATURES_CSV    = f'{OUTPUT_DIR}/feature_matrix.csv'
FEAT_COLS_PKL   = f'{OUTPUT_DIR}/feature_cols.pkl'
SCALER_PKL      = f'{OUTPUT_DIR}/scaler.pkl'
MODEL_PT        = f'{OUTPUT_DIR}/model.pt'
CALIBRATOR_PKL  = f'{OUTPUT_DIR}/calibrator.pkl'
METRICS_JSON    = f'{OUTPUT_DIR}/test_metrics.json'
RANDOM_SEED     = 42
BATCH_SIZE      = 512
TRAIN_NEG_RATIO = 3
TRAIN_RATIO     = 0.70
VAL_RATIO       = 0.15
THRESHOLD_HIGH  = 0.90   # MATCH
THRESHOLD_MID   = 0.70   # POSSIBLE_MATCH
# ──────────────────────────────────────────────────────────────────────────

import os, pickle, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              roc_auc_score, average_precision_score,
                              classification_report, confusion_matrix,
                              roc_curve, precision_recall_curve, brier_score_loss)
from collections import Counter

# Load artifacts
feature_matrix = pd.read_csv(FEATURES_CSV)
with open(FEAT_COLS_PKL, 'rb') as f: feature_cols = pickle.load(f)
with open(SCALER_PKL,    'rb') as f: scaler       = pickle.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Rebuild splits (same seed as Stage 10/11)
unique_entities = feature_matrix['entity_id_a'].dropna().unique()
rng = np.random.default_rng(RANDOM_SEED)
shuffled = unique_entities.copy()
rng.shuffle(shuffled)
n       = len(shuffled)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * (TRAIN_RATIO + VAL_RATIO))
val_entities  = set(shuffled[n_train:n_val])
test_entities = set(shuffled[n_val:])
val_df   = feature_matrix[feature_matrix['entity_id_a'].isin(val_entities)]
test_df  = feature_matrix[feature_matrix['entity_id_a'].isin(test_entities)]

X_val  = scaler.transform(val_df[feature_cols].values)
y_val  = val_df['label'].values.astype(np.float32)
X_test = scaler.transform(test_df[feature_cols].values)
y_test = test_df['label'].values.astype(np.float32)

class PairDataset(Dataset):
    def __init__(self, X, y): self.X=torch.FloatTensor(X); self.y=torch.FloatTensor(y)
    def __len__(self):        return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

val_loader  = DataLoader(PairDataset(X_val,  y_val),  batch_size=BATCH_SIZE)
test_loader = DataLoader(PairDataset(X_test, y_test), batch_size=BATCH_SIZE)

# Load model
class IdentityMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,64),  nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64,1)
        )
    def forward(self, x): return self.network(x).squeeze(-1)

model = IdentityMLP(len(feature_cols)).to(device)
model.load_state_dict(torch.load(MODEL_PT, map_location=device, weights_only=True))
model.eval()
print(f'Model loaded | val: {len(X_val):,} | test: {len(X_test):,}')

Model loaded | val: 30,660 | test: 30,713


/var/folders/ht/_lrx9n5s0539yfctp63z0yz00000gn/T/ipykernel_24341/1936298701.py:45: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(shuffled)


### Step 12.1: Collect Raw Probabilities
ใช้ **validation set** เท่านั้น

In [2]:
# --- 12.1 Collect Raw Probabilities ---
model.eval()
val_probs_raw, val_labels_all = [], []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch  = X_batch.to(device)
        logits   = model(X_batch)
        probs    = torch.sigmoid(logits).cpu().numpy()
        val_probs_raw.extend(probs)
        val_labels_all.extend(y_batch.numpy())

val_probs_raw  = np.array(val_probs_raw)
val_labels_all = np.array(val_labels_all)

print('📊 Step 12.1: Raw Probabilities (Validation Set)')
print('=' * 60)
print(f'  Samples     : {len(val_probs_raw):,}')
print(f'  Prob mean   : {val_probs_raw.mean():.4f}')
print(f'  Prob std    : {val_probs_raw.std():.4f}')
print(f'  Pos prob    : {val_probs_raw[val_labels_all==1].mean():.4f}')
print(f'  Neg prob    : {val_probs_raw[val_labels_all==0].mean():.4f}')
print(f'\n✅ Step 12.1 เสร็จ')

📊 Step 12.1: Raw Probabilities (Validation Set)
  Samples     : 30,660
  Prob mean   : 0.2902
  Prob std    : 0.2470
  Pos prob    : 0.8384
  Neg prob    : 0.1989

✅ Step 12.1 เสร็จ


### Step 12.2: Fit Calibrator

In [3]:
# --- 12.2 Fit Calibrator ---
calibrator     = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(val_probs_raw, val_labels_all)
val_probs_cal  = calibrator.predict(val_probs_raw)

with open(CALIBRATOR_PKL, 'wb') as f:
    pickle.dump(calibrator, f)

print('📊 Step 12.2: Calibrator Fitted')
print('=' * 60)
print(f'  Method       : Isotonic Regression')
print(f'  Before cal   : mean={val_probs_raw.mean():.4f}')
print(f'  After cal    : mean={val_probs_cal.mean():.4f}')
print(f'  Saved: {CALIBRATOR_PKL}')
print(f'\n✅ Step 12.2 เสร็จ')

📊 Step 12.2: Calibrator Fitted
  Method       : Isotonic Regression
  Before cal   : mean=0.2902
  After cal    : mean=0.1428
  Saved: /Users/tm/Documents/GitHub/Project-for-Work/data/processed/calibrator.pkl

✅ Step 12.2 เสร็จ


### Step 12.3: Calibration Quality Check

In [4]:
# --- 12.3 Calibration Quality ---
def expected_calibration_error(probs, labels, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i+1])
        if mask.sum() > 0:
            avg_conf = probs[mask].mean()
            avg_acc  = labels[mask].mean()
            ece     += mask.sum() / len(probs) * abs(avg_conf - avg_acc)
    return ece

ece_before   = expected_calibration_error(val_probs_raw, val_labels_all)
ece_after    = expected_calibration_error(val_probs_cal, val_labels_all)
brier_before = brier_score_loss(val_labels_all, val_probs_raw)
brier_after  = brier_score_loss(val_labels_all, val_probs_cal)

print('=' * 60)
print('📊 STAGE 12 v2 SUMMARY — Calibration')
print('=' * 60)
print(f'  {"Metric":<20} {"Before":<12} {"After":<12} {"Change":<12}')
print(f'  {"-"*56}')
print(f'  {"ECE":<20} {ece_before:<12.4f} {ece_after:<12.4f} {ece_after-ece_before:+.4f}')
print(f'  {"Brier Score":<20} {brier_before:<12.4f} {brier_after:<12.4f} {brier_after-brier_before:+.4f}')

# Reliability diagram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, probs, title in [(ax1, val_probs_raw, 'Before'), (ax2, val_probs_cal, 'After')]:
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers, bin_accs = [], []
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i+1])
        if mask.sum() > 0:
            bin_centers.append(probs[mask].mean())
            bin_accs.append(val_labels_all[mask].mean())
    ax.plot([0,1], [0,1], 'k--', alpha=0.5, label='Perfect')
    ax.plot(bin_centers, bin_accs, 'bo-', label='Model')
    ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Actual Probability')
    ax.set_title(f'Reliability Diagram ({title} Calibration)')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/calibration_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Stage 12 v2 COMPLETE')

📊 STAGE 12 v2 SUMMARY — Calibration
  Metric               Before       After        Change      
  --------------------------------------------------------
  ECE                  0.1623       0.0000       -0.1623
  Brier Score          0.0486       0.0193       -0.0293



✅ Stage 12 v2 COMPLETE


/var/folders/ht/_lrx9n5s0539yfctp63z0yz00000gn/T/ipykernel_24341/1959899453.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Stage 13 v2: Threshold Selection & Final Evaluation

**กฎสำคัญ:** Threshold selection → validation set | Final evaluation → test set ครั้งเดียว!

### Step 13.1: Threshold Selection
เลือก threshold จาก **validation set** เท่านั้น

In [5]:
# --- 13.1 Threshold Selection ---
thresholds = np.arange(0.01, 1.00, 0.01)
f1_scores  = []

for t in thresholds:
    preds = (val_probs_cal >= t).astype(int)
    f1    = f1_score(val_labels_all, preds, zero_division=0)
    f1_scores.append(f1)

best_idx           = np.argmax(f1_scores)
optimal_threshold  = thresholds[best_idx]
best_f1            = f1_scores[best_idx]

print('📊 Step 13.1: Threshold Selection (Validation Set)')
print('=' * 60)
print(f'  Optimal threshold : {optimal_threshold:.2f}')
print(f'  Best F1 score     : {best_f1:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1_scores, 'b-')
plt.axvline(x=optimal_threshold, color='r', linestyle='--', label=f'Optimal={optimal_threshold:.2f}')
plt.xlabel('Threshold'); plt.ylabel('F1 Score')
plt.title('F1 vs Threshold (Validation Set)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig(f'{OUTPUT_DIR}/threshold_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Step 13.1 เสร็จ')

📊 Step 13.1: Threshold Selection (Validation Set)
  Optimal threshold : 0.38
  Best F1 score     : 0.9172

✅ Step 13.1 เสร็จ


/var/folders/ht/_lrx9n5s0539yfctp63z0yz00000gn/T/ipykernel_24341/343277177.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Step 13.2: 3-Level Threshold Definition

In [6]:
# --- 13.2 3-Level Threshold ---
def get_decision(prob):
    if prob >= THRESHOLD_HIGH:  return 'MATCH'
    elif prob >= THRESHOLD_MID: return 'POSSIBLE_MATCH'
    else:                       return 'NO_MATCH'

val_decisions  = [get_decision(p) for p in val_probs_cal]
decision_counts = Counter(val_decisions)

print('📊 Step 13.2: 3-Level Threshold')
print('=' * 60)
print(f'  {"Level":<18} {"Range":<18} {"Action":<18} {"Count":<10}')
print(f'  {"-"*64}')
print(f'  {"MATCH":<18} {">=90%":<18} {"Auto-merge":<18} {decision_counts.get("MATCH",0):<10}')
print(f'  {"POSSIBLE_MATCH":<18} {"70-89%":<18} {"Human review":<18} {decision_counts.get("POSSIBLE_MATCH",0):<10}')
print(f'  {"NO_MATCH":<18} {"<70%":<18} {"Keep separate":<18} {decision_counts.get("NO_MATCH",0):<10}')
print(f'\n✅ Step 13.2 เสร็จ')

📊 Step 13.2: 3-Level Threshold
  Level              Range              Action             Count     
  ----------------------------------------------------------------
  MATCH              >=90%              Auto-merge         3462      
  POSSIBLE_MATCH     70-89%             Human review       287       
  NO_MATCH           <70%               Keep separate      26911     

✅ Step 13.2 เสร็จ


### Step 13.3: Final Evaluation on Test Set
**ใช้ TEST SET ตรงนี้ครั้งเดียว — ห้ามย้อนกลับไปปรับ model!**

In [7]:
# --- 13.3 Final Evaluation (Test Set) ---
model.eval()
test_probs_raw, test_labels_all = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits  = model(X_batch)
        probs   = torch.sigmoid(logits).cpu().numpy()
        test_probs_raw.extend(probs)
        test_labels_all.extend(y_batch.numpy())

test_probs_raw  = np.array(test_probs_raw)
test_labels_all = np.array(test_labels_all)
test_probs_cal  = calibrator.predict(test_probs_raw)
test_preds      = (test_probs_cal >= optimal_threshold).astype(int)

roc_auc  = roc_auc_score(test_labels_all, test_probs_cal)
avg_prec = average_precision_score(test_labels_all, test_probs_cal)

print('=' * 60)
print('📊 STAGE 13 v2 — FINAL EVALUATION (TEST SET)')
print('=' * 60)
print(f'  Threshold    : {optimal_threshold:.2f}')
print(f'  Samples      : {len(test_labels_all):,}')
print(f'  ROC-AUC      : {roc_auc:.4f}')
print(f'  Avg Precision: {avg_prec:.4f}')
print(f'\n{classification_report(test_labels_all, test_preds, target_names=["NO_MATCH","MATCH"])}')

cm = confusion_matrix(test_labels_all, test_preds)
print(f'  Confusion Matrix:')
print(f'    TN={cm[0][0]:,}  FP={cm[0][1]:,}')
print(f'    FN={cm[1][0]:,}  TP={cm[1][1]:,}')

test_decisions      = [get_decision(p) for p in test_probs_cal]
test_decision_counts = Counter(test_decisions)
print(f'\n  3-Level Decision Distribution (Test):')
for level in ['MATCH', 'POSSIBLE_MATCH', 'NO_MATCH']:
    print(f'    {level:<18}: {test_decision_counts.get(level,0):,}')

# ROC + PR curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fpr, tpr, _      = roc_curve(test_labels_all, test_probs_cal)
ax1.plot(fpr, tpr, 'b-', label=f'AUC={roc_auc:.3f}')
ax1.plot([0,1],[0,1],'k--',alpha=0.3)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curve (Test Set v2)'); ax1.legend(); ax1.grid(True, alpha=0.3)
prec_arr, rec_arr, _ = precision_recall_curve(test_labels_all, test_probs_cal)
ax2.plot(rec_arr, prec_arr, 'r-', label=f'AP={avg_prec:.3f}')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('PR Curve (Test Set v2)'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_pr_v2.png', dpi=150, bbox_inches='tight')
plt.show()

# Save metrics
metrics = {
    'version':       'v2',
    'threshold':     float(optimal_threshold),
    'roc_auc':       float(roc_auc),
    'avg_precision': float(avg_prec),
    'precision':     float(precision_score(test_labels_all, test_preds, zero_division=0)),
    'recall':        float(recall_score(test_labels_all, test_preds, zero_division=0)),
    'f1':            float(f1_score(test_labels_all, test_preds, zero_division=0)),
    'decisions': {
        'match':          test_decision_counts.get('MATCH', 0),
        'possible_match': test_decision_counts.get('POSSIBLE_MATCH', 0),
        'no_match':       test_decision_counts.get('NO_MATCH', 0),
    }
}
with open(METRICS_JSON, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\n  Saved: {METRICS_JSON}')
print(f'\n{"="*60}')
print(f'✅ Stage 13 v2 COMPLETE — Final evaluation done')
print(f'{"="*60}')

📊 STAGE 13 v2 — FINAL EVALUATION (TEST SET)
  Threshold    : 0.38
  Samples      : 30,713
  ROC-AUC      : 0.9702
  Avg Precision: 0.9414

              precision    recall  f1-score   support

    NO_MATCH       0.98      0.99      0.99     26323
       MATCH       0.96      0.87      0.91      4390

    accuracy                           0.98     30713
   macro avg       0.97      0.93      0.95     30713
weighted avg       0.98      0.98      0.98     30713

  Confusion Matrix:
    TN=26,160  FP=163
    FN=554  TP=3,836

  3-Level Decision Distribution (Test):
    MATCH             : 3,422
    POSSIBLE_MATCH    : 316
    NO_MATCH          : 26,975


/var/folders/ht/_lrx9n5s0539yfctp63z0yz00000gn/T/ipykernel_24341/3251403216.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



  Saved: /Users/tm/Documents/GitHub/Project-for-Work/data/processed/test_metrics.json

✅ Stage 13 v2 COMPLETE — Final evaluation done
